In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.data.api_client import APIClientFactory
from src.data.data_validator import DataValidator

In [3]:
import pandas as pd
import numpy as np

In [4]:
client = APIClientFactory.get_primary_client()
validator = DataValidator()

In [5]:
historical_aqi = client.fetch_historical("Lahore", 31.558, 74.351, "2026-07-20", "2026-07-25")

2026-07-26 22:45:49 | INFO     | OpenMeteoClient:75 | Fetching historical data from https://air-quality-api.open-meteo.com/v1/air-quality for Lahore [2026-07-20 -> 2026-07-25]


In [6]:
df_aqi = pd.DataFrame(historical_aqi)

In [39]:
df = df_aqi.copy()

## Features Generation

#### Temporal Features

These are usually human-readable features extracted from timestamp and information like the high AQI on weekdays or weekends will help identify the pattern.

In [40]:
# 1. Temporal features
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

#### Cyclic Encoding

Keep continuos and cyclic nature of timely features like looping of time and months. Example like 23 hour (11pm) is far from 0 hour but actually on clock sit right next to each other.
</br>
Trigonometric functions are used to keep such nature.  

In [41]:
# 2. Cyclical encoding 
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [42]:
# Group once, reused for all per-city time-series ops below
g = df.groupby("city")["european_aqi"]

#### Lag Hours

Historical values of target variable from previous time steps as it provides historical context and autocorrelation like what happens recently will predicts what will happen next

In [43]:
LAG_HOURS = [1, 3]
# 3. Lag features
for lag in LAG_HOURS:
    df[f"aqi_lag_{lag}h"] = g.shift(lag)

#### Rolling windows

Aggregated mean and standard deviation on across a sliding window of historical rows. Rolling mean smooth the noise and rolling Std tells the volatility or stability of signal. It tells the model the consistency and spikes in the aqi trend 

In [44]:
ROLLING_WINDOWS = [6, 24]
# 4. Rolling statistics (shift(1) first so window never includes current row)

for window in ROLLING_WINDOWS:
    df[f"aqi_roll_mean_{window}h"] = (
        df.groupby("city")["european_aqi"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )
    df[f"aqi_roll_std_{window}h"] = (
        df.groupby("city")["european_aqi"]
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

#### Rate of Change

Delta between previous step and current value expose the momentum and direction over short and long horizons. Example AQI reading of 80 that was 30 an hour ago tells that air become more pollutant.

In [45]:
# 5. Rate of change
df["aqi_change_1h"] = g.shift(1).diff(1) # keep it to t-1
df["aqi_change_24h"] = g.shift(1).diff(24)

#### Target Variable

In [46]:
forecast_horizon = 1

In [47]:
# 7. Target creation (shift AQI forward, per city)
for day in range(1, forecast_horizon + 1):
    df[f"aqi_next_{day}d"] = df.groupby("city")["european_aqi"].shift(-day * 24)

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype                       
---  ------                 --------------  -----                       
 0   date                   144 non-null    datetime64[ns, Asia/Karachi]
 1   city                   144 non-null    object                      
 2   lat                    144 non-null    float64                     
 3   lon                    144 non-null    float64                     
 4   pm10                   144 non-null    float64                     
 5   pm2_5                  144 non-null    float64                     
 6   carbon_monoxide        144 non-null    float64                     
 7   nitrogen_dioxide       144 non-null    float64                     
 8   sulphur_dioxide        144 non-null    float64                     
 9   ozone                  144 non-null    float64                     
 10  uv_index      

In [49]:
df.describe()

,lat,lon,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,uv_index,aerosol_optical_depth,...,month_cos,aqi_lag_1h,aqi_lag_3h,aqi_roll_mean_6h,aqi_roll_std_6h,aqi_roll_mean_24h,aqi_roll_std_24h,aqi_change_1h,aqi_change_24h,aqi_next_1d
count,144.000,144.000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,...,1.440000e+02,143.000000,141.000000,138.000000,138.000000,120.000000,120.000000,142.000000,119.000000,120.000000
mean,31.558,74.351,60.119445,39.950695,540.277778,21.786111,8.836111,84.430556,1.178472,0.932153,...,-8.660254e-01,76.866547,77.004654,76.378348,1.857421,73.868237,5.810740,-0.326584,-9.610449,68.757301
std,0.000,0.000,63.254859,24.854891,182.040972,12.656564,2.612573,39.895432,1.818280,0.334591,...,2.228196e-16,21.842355,21.966554,21.519385,1.759986,18.757805,4.627104,1.955708,19.883718,13.322531
min,31.558,74.351,11.400000,9.300000,263.000000,3.900000,2.800000,19.000000,0.000000,0.470000,...,-8.660254e-01,45.883339,45.883339,47.866671,0.100500,57.302187,0.564431,-7.575760,-47.934998,45.883339
25%,31.558,74.351,23.275000,21.975000,422.000000,13.200000,7.300000,55.000000,0.000000,0.660000,...,-8.660254e-01,63.014999,63.013336,62.964862,0.483753,62.700174,2.383914,-0.990831,-26.268337,62.835000
50%,31.558,74.351,40.049999,33.000000,484.500000,19.500000,8.400000,73.000000,0.100000,0.865000,...,-8.660254e-01,67.353333,67.353333,66.974949,1.410877,64.799730,4.141950,-0.196669,-0.820000,64.763336
75%,31.558,74.351,61.200000,47.550000,616.000000,28.050000,9.825000,115.000000,1.762500,1.110000,...,-8.660254e-01,88.409164,90.603333,84.941113,2.714558,78.634342,8.695035,0.309244,3.645004,69.888332
max,31.558,74.351,323.200012,119.000000,1186.000000,70.400002,17.500000,182.000000,7.400000,2.230000,...,-8.660254e-01,122.641678,122.641678,121.604730,7.213756,116.965835,15.847391,16.159084,17.526657,112.118340


In [50]:
df.loc[:, df.isna().any()]

,aqi_lag_1h,aqi_lag_3h,aqi_roll_mean_6h,aqi_roll_std_6h,aqi_roll_mean_24h,aqi_roll_std_24h,aqi_change_1h,aqi_change_24h,aqi_next_1d
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.118340
1,113.241653,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.488327
2,113.093330,NaN,NaN,NaN,NaN,NaN,-0.148323,NaN,108.833328
3,113.000000,113.241653,NaN,NaN,NaN,NaN,-0.093330,NaN,107.099998
4,113.348328,113.093330,NaN,NaN,NaN,NaN,0.348328,NaN,105.660004
...,...,...,...,...,...,...,...,...,...
139,68.386673,68.663338,68.913335,0.458249,68.920001,2.053497,-0.226662,3.720009,NaN
140,68.216667,68.613335,68.679447,0.376223,69.068751,1.849674,-0.170006,3.570000,NaN
141,67.843330,68.386673,68.443335,0.383477,69.191112,1.648572,-0.373337,2.936661,NaN
142,67.393333,68.216667,68.186113,0.489371,69.275973,1.487124,-0.449997,2.036667,NaN


In [51]:
df_ok =  df.dropna()

In [52]:
df_ok

,date,city,lat,lon,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,...,month_cos,aqi_lag_1h,aqi_lag_3h,aqi_roll_mean_6h,aqi_roll_std_6h,aqi_roll_mean_24h,aqi_roll_std_24h,aqi_change_1h,aqi_change_24h,aqi_next_1d
25,2026-07-21 01:00:00+05:00,Lahore,31.558,74.351,56.200001,41.200001,464.0,20.100000,8.2,68.0,...,-0.866025,112.118340,115.111679,115.507228,2.342915,116.919031,3.412586,-1.556664,-1.123314,65.243332
26,2026-07-21 02:00:00+05:00,Lahore,31.558,74.351,63.599998,42.000000,423.0,19.900000,8.9,61.0,...,-0.866025,110.488327,113.675003,114.200003,2.623170,116.810489,3.577024,-1.630013,-2.605003,64.683334
27,2026-07-21 03:00:00+05:00,Lahore,31.558,74.351,70.099998,41.599998,376.0,20.100000,9.5,54.0,...,-0.866025,108.833328,112.118340,112.765835,2.841776,116.636877,3.859935,-1.654999,-4.166672,64.083336
28,2026-07-21 04:00:00+05:00,Lahore,31.558,74.351,78.699997,43.000000,328.0,20.600000,10.2,48.0,...,-0.866025,107.099998,110.488327,111.221113,3.006166,116.376530,4.279323,-1.733330,-6.248329,63.480000
29,2026-07-21 05:00:00+05:00,Lahore,31.558,74.351,130.399994,47.700001,309.0,17.000000,11.0,58.0,...,-0.866025,105.660004,108.833328,109.645833,3.035970,116.041947,4.782803,-1.439995,-8.029991,62.836666
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2026-07-24 19:00:00+05:00,Lahore,31.558,74.351,52.500000,52.299999,961.0,44.299999,9.8,81.0,...,-0.866025,64.666664,67.090904,67.281968,2.110912,63.792652,2.498108,-0.176674,1.673328,68.216667
116,2026-07-24 20:00:00+05:00,Lahore,31.558,74.351,55.500000,55.400002,1137.0,58.700001,10.6,50.0,...,-0.866025,64.646667,64.843338,66.601868,2.207280,63.860569,2.498252,-0.019997,1.630005,67.843330
117,2026-07-24 21:00:00+05:00,Lahore,31.558,74.351,58.599998,58.299999,1186.0,66.599998,11.7,35.0,...,-0.866025,64.906670,64.666664,65.843889,1.769134,63.939458,2.500228,0.260002,1.893333,67.393333
118,2026-07-24 22:00:00+05:00,Lahore,31.558,74.351,64.300003,64.300003,1160.0,70.400002,13.0,30.0,...,-0.866025,65.356667,64.646667,65.251818,0.936785,64.041263,2.506366,0.449997,2.443329,66.866669
